# Tesla EA Deliveries & Production Data (2015–2025)
## End-to-End ML Pipeline: Preprocessing → EDA → Feature Engineering → Regression → Time Series Forecasting

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load Data

In [ ]:
df = pd.read_csv('/kaggle/input/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv')
print('Shape:', df.shape)
df.head()

## 2. Preprocessing

In [ ]:
# ── Basic Info ─────────────────────────────────────────────────────────────
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Duplicates ---')
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# ── Handle Missing Values ──────────────────────────────────────────────────
# Fill numeric columns with median
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Drop duplicates
df.drop_duplicates(inplace=True)

print('After cleaning — Shape:', df.shape)
print('Missing values remaining:', df.isnull().sum().sum())

In [ ]:
# ── Statistical Summary ────────────────────────────────────────────────────
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Deliveries by Year ─────────────────────────────────────────────────────
yearly = df.groupby('Year')['Estimated_Deliveries'].sum().reset_index()

plt.figure(figsize=(12,5))
sns.barplot(data=yearly, x='Year', y='Estimated_Deliveries', palette='Blues_d')
plt.title('Total Estimated Deliveries by Year (2015–2025)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Total Deliveries')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Deliveries by Region ───────────────────────────────────────────────────
region_del = df.groupby('Region')['Estimated_Deliveries'].sum().reset_index().sort_values('Estimated_Deliveries', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=region_del, x='Region', y='Estimated_Deliveries', palette='Set2')
plt.title('Total Deliveries by Region', fontsize=14)
plt.xlabel('Region')
plt.ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# ── Deliveries by Model ────────────────────────────────────────────────────
model_del = df.groupby('Model')['Estimated_Deliveries'].sum().reset_index().sort_values('Estimated_Deliveries', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=model_del, x='Model', y='Estimated_Deliveries', palette='rocket')
plt.title('Total Deliveries by Model', fontsize=14)
plt.xlabel('Model')
plt.ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────
plt.figure(figsize=(10,7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution of Estimated Deliveries ──────────────────────────────────
plt.figure(figsize=(8,4))
sns.histplot(df['Estimated_Deliveries'], bins=40, kde=True, color='steelblue')
plt.title('Distribution of Estimated Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# ── Pairplot of Key Numeric Features ──────────────────────────────────────
key_cols = ['Estimated_Deliveries'] + [c for c in num_cols if c not in ['Year','Month','Estimated_Deliveries']][:3]
sns.pairplot(df[key_cols], diag_kind='kde', plot_kws={'alpha':0.4})
plt.suptitle('Pairplot of Key Numerical Features', y=1.02)
plt.show()

## 4. Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_fe = df.copy()

# ── Date Features ──────────────────────────────────────────────────────────
df_fe['Quarter'] = ((df_fe['Month'] - 1) // 3) + 1
df_fe['Is_Q4']   = (df_fe['Quarter'] == 4).astype(int)
df_fe['YearMonth'] = df_fe['Year'] * 100 + df_fe['Month']   # ordinal time index

# ── Cyclical Encoding for Month ────────────────────────────────────────────
df_fe['Month_sin'] = np.sin(2 * np.pi * df_fe['Month'] / 12)
df_fe['Month_cos'] = np.cos(2 * np.pi * df_fe['Month'] / 12)

# ── Label Encoding for Categoricals ───────────────────────────────────────
le = LabelEncoder()
for col in cat_cols:
    df_fe[col + '_enc'] = le.fit_transform(df_fe[col])

# ── Rolling Average (lag feature proxy) ───────────────────────────────────
df_fe = df_fe.sort_values(['Year','Month'])
df_fe['Delivery_RollMean3'] = df_fe['Estimated_Deliveries'].rolling(window=3, min_periods=1).mean()

print('New features added:', [c for c in df_fe.columns if c not in df.columns])
df_fe.head()

## 5. Regression Modeling + Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Feature Selection ──────────────────────────────────────────────────────
feature_cols = [
    'Year', 'Month', 'Quarter', 'Is_Q4',
    'Month_sin', 'Month_cos', 'Delivery_RollMean3'
] + [c for c in df_fe.columns if c.endswith('_enc')] \
  + [c for c in num_cols if c not in ['Year','Month','Estimated_Deliveries']]

# Keep only columns that actually exist
feature_cols = [c for c in feature_cols if c in df_fe.columns]

X = df_fe[feature_cols]
y = df_fe['Estimated_Deliveries']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape} | Test size: {X_test.shape}')

In [ ]:
# ── Baseline: Ridge Regression ─────────────────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

print('=== Ridge Regression ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_ridge):.2f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.2f}')
print(f'R²   : {r2_score(y_test, y_pred_ridge):.4f}')

In [ ]:
# ── Random Forest ──────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_rf):.2f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}')
print(f'R²   : {r2_score(y_test, y_pred_rf):.4f}')

In [ ]:
# ── Gradient Boosting ──────────────────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print('=== Gradient Boosting ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_gb):.2f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.2f}')
print(f'R²   : {r2_score(y_test, y_pred_gb):.4f}')

In [ ]:
# ── Hyperparameter Tuning (GridSearchCV on Random Forest) ──────────────────
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print('Best Params:', grid_search.best_params_)
print('Best CV R²:', round(grid_search.best_score_, 4))

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print(f'\nTuned RF → MAE: {mean_absolute_error(y_test, y_pred_best):.2f} | R²: {r2_score(y_test, y_pred_best):.4f}')

In [ ]:
# ── Model Comparison Chart ─────────────────────────────────────────────────
models = ['Ridge', 'Random Forest', 'Gradient Boosting', 'Tuned RF']
r2_scores = [
    r2_score(y_test, y_pred_ridge),
    r2_score(y_test, y_pred_rf),
    r2_score(y_test, y_pred_gb),
    r2_score(y_test, y_pred_best)
]

plt.figure(figsize=(8,4))
bars = plt.bar(models, r2_scores, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
plt.ylim(0, 1.1)
plt.ylabel('R² Score')
plt.title('Model Comparison — R² Score')
for bar, val in zip(bars, r2_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ─────────────────────────────────────────────────────
feat_imp = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10,5))
feat_imp.head(15).plot(kind='bar', color='steelblue')
plt.title('Top 15 Feature Importances (Tuned Random Forest)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 6. Time Series Forecasting

In [ ]:
# ── Aggregate monthly deliveries (global) ─────────────────────────────────
ts = df.groupby(['Year','Month'])['Estimated_Deliveries'].sum().reset_index()
ts['Date'] = pd.to_datetime(ts[['Year','Month']].assign(day=1))
ts = ts.sort_values('Date').set_index('Date')
ts_series = ts['Estimated_Deliveries']

plt.figure(figsize=(14,5))
plt.plot(ts_series, marker='o', markersize=3, linewidth=1.5, color='steelblue')
plt.title('Monthly Estimated Deliveries Over Time')
plt.xlabel('Date')
plt.ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# ── SARIMA Forecasting ─────────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# ADF Test for stationarity
adf_result = adfuller(ts_series)
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print('Stationary    :', 'Yes' if adf_result[1] < 0.05 else 'No (may need differencing)')

In [ ]:
# ── Train SARIMA(1,1,1)(1,1,1,12) ──────────────────────────────────────────
train_ts = ts_series[:-12]
test_ts  = ts_series[-12:]
ts_series = ts['Estimated_Deliveries'].asfreq('MS')

sarima_model = SARIMAX(
    train_ts,
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

In [ ]:
# ── Forecast & Evaluate ────────────────────────────────────────────────────
forecast = sarima_fit.forecast(steps=12)
forecast.index = test_ts.index

print('=== SARIMA Forecast Metrics ===')
print(f'MAE  : {mean_absolute_error(test_ts, forecast):.2f}')
print(f'RMSE : {np.sqrt(mean_squared_error(test_ts, forecast)):.2f}')
print(f'R²   : {r2_score(test_ts, forecast):.4f}')

plt.figure(figsize=(14,5))
plt.plot(train_ts, label='Train', color='steelblue')
plt.plot(test_ts,  label='Actual (Test)', color='green')
plt.plot(forecast, label='SARIMA Forecast', color='red', linestyle='--')
plt.title('SARIMA: Actual vs Forecast (Last 12 Months)')
plt.xlabel('Date')
plt.ylabel('Deliveries')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Future 12-Month Forecast ───────────────────────────────────────────────
future_model = SARIMAX(
    ts_series,
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

future_forecast = future_model.forecast(steps=12)

plt.figure(figsize=(14,5))
plt.plot(ts_series, label='Historical', color='steelblue')
plt.plot(future_forecast, label='Future Forecast (2025–2026)', color='orange', linestyle='--', marker='o')
plt.title('Tesla Deliveries — Future 12-Month Forecast')
plt.xlabel('Date')
plt.ylabel('Deliveries')
plt.legend()
plt.tight_layout()
plt.show()

print('\nForecasted Monthly Deliveries:')
print(future_forecast.round(0))

## Summary

| Step | Method | Notes |
|------|--------|-------|
| Preprocessing | Median/Mode imputation, dedup | Clean dataset |
| EDA | Bar plots, heatmap, pairplot | Trends by year/region/model |
| Feature Engineering | Cyclical encoding, rolling mean, label encoding | 7+ new features |
| Regression | Ridge, Random Forest, Gradient Boosting | GridSearchCV tuning |
| Time Series | SARIMA(1,1,1)(1,1,1,12) | 12-month forecast |